# Barter-RS: Engine, Strategies, Risk & Statistics

This notebook covers the full Rust engine pipeline: configuration,
backtesting, custom strategies, risk management, and trading statistics.

## Topics Covered
- `SystemConfig` JSON structure and `SystemBuilder` lifecycle
- Engine architecture: clock, strategy, risk manager, execution
- The 4 strategy traits and multi-strategy composition
- Custom `RiskManager` with composable checks
- `TradingSummaryGenerator` for performance analysis
- Annualised metrics: Sharpe, Sortino, Calmar, max drawdown


In [ ]:
:dep barter = { version = "0.12" }
:dep barter-data = { version = "0.11" }
:dep barter-execution = { version = "0.7" }
:dep barter-instrument = { version = "0.3" }
:dep barter-integration = { version = "0.10" }
:dep tokio = { version = "1", features = ["full"] }
:dep futures = "0.3"
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"
:dep rust_decimal = { version = "1.36", features = ["maths"] }
:dep rust_decimal_macros = "1.29"
:dep smol_str = "0.3"
:dep chrono = { version = "0.4", features = ["serde"] }
:dep itertools = "0.14"
:dep tracing = "0.1"
:dep tracing-subscriber = { version = "0.3", features = ["env-filter", "json"] }


---
## Part 1: Engine & Backtesting


## 1. System Configuration

Barter uses a JSON-based `SystemConfig` that defines:
- **Instruments**: which exchange/asset pairs to trade
- **Executions**: mock or live execution clients with initial balances

Here's the structure of a typical config:

In [ ]:
let config_json = r#"{
  "instruments": [
    {
      "exchange": "binance_spot",
      "name_exchange": "BTCUSDT",
      "underlying": { "base": "btc", "quote": "usdt" },
      "quote": "underlying_quote",
      "kind": "spot"
    },
    {
      "exchange": "binance_spot",
      "name_exchange": "ETHUSDT",
      "underlying": { "base": "eth", "quote": "usdt" },
      "quote": "underlying_quote",
      "kind": "spot"
    }
  ],
  "executions": [
    {
      "mocked_exchange": "binance_spot",
      "latency_ms": 100,
      "fees_percent": 0.05,
      "initial_state": {
        "exchange": "binance_spot",
        "balances": [
          { "asset": "usdt", "balance": { "total": 10000, "free": 10000 }, "time_exchange": "2025-01-01T00:00:00Z" },
          { "asset": "btc", "balance": { "total": 0.1, "free": 0.1 }, "time_exchange": "2025-01-01T00:00:00Z" },
          { "asset": "eth", "balance": { "total": 1.0, "free": 1.0 }, "time_exchange": "2025-01-01T00:00:00Z" }
        ],
        "instruments": [
          { "instrument": "BTCUSDT", "orders": [] },
          { "instrument": "ETHUSDT", "orders": [] }
        ]
      }
    }
  ]
}"#;

println!("Config JSON defined with 2 instruments and mock execution.");
println!("Starting balance: 10,000 USDT + 0.1 BTC + 1.0 ETH");

## 2. Building the System

The `SystemBuilder` wires together all components:
- **Clock**: `LiveClock` for real-time, `HistoricalClock` for backtesting
- **Strategy**: implements `AlgoStrategy` + `ClosePositionsStrategy` + `OnDisconnectStrategy`
- **RiskManager**: filters/approves orders before execution
- **EngineFeedMode**: `Iterator` (sync) or `Stream` (async)
- **AuditMode**: `Enabled` to replicate state, `Disabled` for performance

In [ ]:
use barter::{
    EngineEvent,
    engine::{
        clock::HistoricalClock,
        state::{
            global::DefaultGlobalData,
            instrument::{data::DefaultInstrumentMarketData, filter::InstrumentFilter},
            trading::TradingState,
        },
    },
    risk::DefaultRiskManager,
    statistic::time::Daily,
    strategy::DefaultStrategy,
    system::{
        builder::{AuditMode, EngineFeedMode, SystemArgs, SystemBuilder},
        config::SystemConfig,
    },
};
use barter_data::streams::{
    consumer::{MarketStreamEvent, MarketStreamResult},
    reconnect::{Event, stream::ReconnectingStream},
};
use barter_data::event::DataKind;
use barter_instrument::{index::IndexedInstruments, instrument::InstrumentIndex};
use futures::{Stream, StreamExt, stream};
use rust_decimal_macros::dec;
use std::time::Duration;

// Parse the config
let config: SystemConfig = serde_json::from_str(config_json)
    .expect("failed to parse SystemConfig");

let instruments = IndexedInstruments::new(config.instruments);
println!("Loaded {} instruments from config", instruments.instruments().len());

// Note: In a real backtest, you would load historical market data from a file or database.
// Here we create an empty stream for demonstration.
let empty_events: Vec<MarketStreamResult<InstrumentIndex, DataKind>> = vec![];
let clock = HistoricalClock::new(chrono::Utc::now());
let market_stream = stream::iter(empty_events)
    .with_error_handler(|error| eprintln!("Stream error: {error:?}"));

println!("Historical clock and market stream prepared.");

## 3. System Lifecycle

The typical lifecycle of a Barter system:

```
SystemArgs::new(...)          // Wire components
  → SystemBuilder::new(args)  // Configure options
  → .build()?                 // Validate and construct
  → .init_with_runtime(...)   // Spawn async tasks
  → system.trading_state(Enabled)  // Start trading
  → ... run ...
  → system.cancel_orders(...)      // Pre-shutdown
  → system.close_positions(...)    // Flatten positions
  → system.shutdown()              // Clean shutdown
  → engine.trading_summary_generator(rfr).generate(Daily)
```

In [ ]:
// Construct SystemArgs
let args = SystemArgs::new(
    &instruments,
    config.executions,
    clock,
    DefaultStrategy::default(),
    DefaultRiskManager::default(),
    market_stream,
    DefaultGlobalData::default(),
    |_| DefaultInstrumentMarketData::default(),
);

// Build the system
let system = SystemBuilder::new(args)
    .engine_feed_mode(EngineFeedMode::Stream)
    .audit_mode(AuditMode::Disabled)
    .trading_state(TradingState::Enabled)
    .build::<EngineEvent, _>()
    .expect("failed to build system")
    .init_with_runtime(tokio::runtime::Handle::current())
    .await
    .expect("failed to init system");

println!("System built and running!");

// Let it process (with our empty stream, it will finish quickly)
tokio::time::sleep(Duration::from_secs(1)).await;

// Graceful shutdown sequence
system.cancel_orders(InstrumentFilter::None);
system.close_positions(InstrumentFilter::None);

let (engine, _) = system.shutdown().await.expect("shutdown failed");

// Generate daily trading summary
let summary = engine
    .trading_summary_generator(dec!(0.05))
    .generate(Daily);

summary.print_summary();
println!("\nEngine shut down successfully.");

## Engine Architecture Overview

```
┌─────────────────┐     ┌──────────────────┐
│  Market Data     │────▶│                  │
│  (barter-data)   │     │     Engine        │────▶ AuditStream
│                  │     │   (Processor)      │       (optional)
└─────────────────┘     │                    │
                        │  ┌──────────────┐  │
┌─────────────────┐     │  │ EngineState   │  │
│  Account Events  │────▶│  │  - positions  │  │
│  (execution rx)  │     │  │  - balances   │  │
│                  │     │  │  - orders     │  │
└─────────────────┘     │  └──────────────┘  │
                        │                    │
┌─────────────────┐     │  ┌──────────────┐  │     ┌──────────────┐
│  Commands        │────▶│  │ Strategy     │──│────▶│  Execution   │
│  (external ctrl) │     │  │ RiskManager  │  │     │  (barter-    │
│                  │     │  └──────────────┘  │     │   execution) │
└─────────────────┘     └──────────────────┘     └──────────────┘
```

### Key Components

| Component | Trait | Purpose |
|-----------|-------|---------|
| Strategy | `AlgoStrategy` | Generate order requests from market state |
| Strategy | `ClosePositionsStrategy` | Generate orders to flatten all positions |
| Strategy | `OnDisconnectStrategy` | React to exchange disconnections |
| RiskManager | `RiskManager` | Approve/reject/modify orders before execution |
| Clock | `EngineClock` | `LiveClock` (real-time) or `HistoricalClock` (backtest) |
| Execution | `ExecutionClient` | Submit orders to exchange (mock or live) |

---
## Part 2: Custom Trading Strategies


## 1. Strategy Trait Overview

Every strategy must implement 4 traits. Here's what each one does:

| Trait | When Called | Purpose |
|-------|------------|---------|
| `AlgoStrategy` | Every engine tick (market data or account event) | Generate order requests (opens & cancels) |
| `ClosePositionsStrategy` | On shutdown or explicit close command | Generate orders to flatten all open positions |
| `OnDisconnectStrategy` | When an exchange WebSocket disconnects | React to connectivity loss (e.g., cancel orders) |
| `OnTradingDisabled` | When trading state transitions to Disabled | Clean up when trading is turned off |

For simple strategies, `OnDisconnectStrategy` and `OnTradingDisabled` can be no-ops.

In [ ]:
use barter::{
    engine::{
        Engine, Processor,
        state::{
            EngineState,
            global::DefaultGlobalData,
            instrument::{
                data::{DefaultInstrumentMarketData, InstrumentDataState},
                filter::InstrumentFilter,
            },
            order::in_flight_recorder::InFlightRequestRecorder,
        },
    },
    strategy::{
        algo::AlgoStrategy,
        close_positions::{ClosePositionsStrategy, build_ioc_market_order_to_close_position},
        on_disconnect::OnDisconnectStrategy,
        on_trading_disabled::OnTradingDisabled,
    },
};
use barter_data::event::{DataKind, MarketEvent};
use barter_execution::{
    AccountEvent,
    order::{
        id::{ClientOrderId, StrategyId},
        request::{OrderRequestCancel, OrderRequestOpen},
    },
};
use barter_instrument::{
    asset::AssetIndex,
    exchange::{ExchangeId, ExchangeIndex},
    instrument::InstrumentIndex,
};
use rust_decimal::Decimal;
use smol_str::SmolStr;

println!("Strategy traits imported.");

## 2. Minimal Strategy Implementation

The simplest possible strategy that does nothing (equivalent to `DefaultStrategy`).
This is a good starting template.

In [ ]:
/// A no-op strategy that never generates orders.
#[derive(Debug, Default)]
struct NoOpStrategy;

// Core: generate order requests each tick
impl AlgoStrategy for NoOpStrategy {
    type State = EngineState<DefaultGlobalData, DefaultInstrumentMarketData>;

    fn generate_algo_orders(
        &self,
        _state: &Self::State, // full engine state is available here
    ) -> (
        impl IntoIterator<Item = OrderRequestCancel<ExchangeIndex, InstrumentIndex>>,
        impl IntoIterator<Item = OrderRequestOpen<ExchangeIndex, InstrumentIndex>>,
    ) {
        // Return empty iterators = no orders
        (std::iter::empty(), std::iter::empty())
    }
}

// Shutdown: generate orders to close all positions
impl ClosePositionsStrategy for NoOpStrategy {
    type State = EngineState<DefaultGlobalData, DefaultInstrumentMarketData>;

    fn close_positions_requests<'a>(
        &'a self,
        state: &'a Self::State,
        filter: &'a InstrumentFilter,
    ) -> (
        impl IntoIterator<Item = OrderRequestCancel> + 'a,
        impl IntoIterator<Item = OrderRequestOpen> + 'a,
    )
    where
        ExchangeIndex: 'a,
        AssetIndex: 'a,
        InstrumentIndex: 'a,
    {
        // No positions to close in this simple strategy
        (std::iter::empty(), std::iter::empty())
    }
}

// Connectivity: react to exchange disconnections
impl<Clock, State, ExecutionTxs, Risk> OnDisconnectStrategy<Clock, State, ExecutionTxs, Risk>
    for NoOpStrategy
{
    type OnDisconnect = ();

    fn on_disconnect(
        _engine: &mut Engine<Clock, State, ExecutionTxs, Self, Risk>,
        _exchange: ExchangeId,
    ) -> Self::OnDisconnect {
        // Could cancel open orders on the disconnected exchange here
    }
}

// Trading control: react when trading is disabled
impl<Clock, State, ExecutionTxs, Risk> OnTradingDisabled<Clock, State, ExecutionTxs, Risk>
    for NoOpStrategy
{
    type OnTradingDisabled = ();

    fn on_trading_disabled(
        _engine: &mut Engine<Clock, State, ExecutionTxs, Self, Risk>,
    ) -> Self::OnTradingDisabled {
        // Could flush state, cancel orders, etc.
    }
}

println!("NoOpStrategy defined with all 4 required trait implementations.");

## 3. Multi-Strategy Composition

A common pattern is running multiple sub-strategies in a single engine.
Each sub-strategy gets its own `StrategyId` and position tracking.

The key idea:
1. Define a `MultiStrategy` struct containing sub-strategies
2. Create custom `InstrumentData` that tracks per-strategy positions
3. Implement `AlgoStrategy` by chaining sub-strategy outputs
4. Use `StrategyId` to tag orders so fills route to the correct sub-strategy

In [ ]:
use barter::engine::state::position::PositionManager;
use barter::statistic::summary::instrument::TearSheetGenerator;
use barter_execution::AccountEventKind;
use chrono::{DateTime, Utc};

/// Multi-strategy wrapper that composes StrategyA and StrategyB
#[derive(Debug, Default)]
struct MultiStrategy {
    strategy_a: StrategyA,
    strategy_b: StrategyB,
}

#[derive(Debug, Default)]
struct StrategyA;
impl StrategyA {
    const ID: StrategyId = StrategyId(SmolStr::new_static("strategy_a"));
}

#[derive(Debug, Default)]
struct StrategyB;
impl StrategyB {
    const ID: StrategyId = StrategyId(SmolStr::new_static("strategy_b"));
}

/// Per-strategy instrument data: each sub-strategy tracks its own positions
#[derive(Debug, Clone)]
struct PerStrategyData {
    tear: TearSheetGenerator,
    position: PositionManager,
}

/// Custom instrument data combining market data + per-strategy state
#[derive(Debug, Clone)]
struct MultiStrategyInstrumentData {
    market_data: DefaultInstrumentMarketData,
    strategy_a: PerStrategyData,
    strategy_b: PerStrategyData,
}

println!("Multi-strategy types defined.");
println!("  StrategyA ID: {}", StrategyA::ID.0);
println!("  StrategyB ID: {}", StrategyB::ID.0);

## 4. Implementing Traits for Custom Instrument Data

Custom instrument data must implement several traits so the engine can
process events through it:

In [ ]:
// InstrumentDataState: exposes the current market price
impl InstrumentDataState for MultiStrategyInstrumentData {
    type MarketEventKind = DataKind;

    fn price(&self) -> Option<Decimal> {
        self.market_data.price()
    }
}

// Process market events: delegate to inner DefaultInstrumentMarketData
impl<InstrumentKey> Processor<&MarketEvent<InstrumentKey, DataKind>>
    for MultiStrategyInstrumentData
{
    type Audit = ();

    fn process(&mut self, event: &MarketEvent<InstrumentKey, DataKind>) -> Self::Audit {
        self.market_data.process(event)
    }
}

// Process account events: route trades to the correct sub-strategy by StrategyId
impl Processor<&AccountEvent> for MultiStrategyInstrumentData {
    type Audit = ();

    fn process(&mut self, event: &AccountEvent) -> Self::Audit {
        let AccountEventKind::Trade(trade) = &event.kind else {
            return;
        };

        // Route trade to the correct strategy based on StrategyId
        if trade.strategy == StrategyA::ID {
            self.strategy_a
                .position
                .update_from_trade(trade)
                .inspect(|closed| self.strategy_a.tear.update_from_position(closed));
        }

        if trade.strategy == StrategyB::ID {
            self.strategy_b
                .position
                .update_from_trade(trade)
                .inspect(|closed| self.strategy_b.tear.update_from_position(closed));
        }
    }
}

// Track in-flight order requests (for duplicate detection)
impl InFlightRequestRecorder for MultiStrategyInstrumentData {
    fn record_in_flight_cancel(&mut self, _: &OrderRequestCancel<ExchangeIndex, InstrumentIndex>) {}
    fn record_in_flight_open(&mut self, _: &OrderRequestOpen<ExchangeIndex, InstrumentIndex>) {}
}

println!("All required trait impls defined for MultiStrategyInstrumentData.");

## 5. Composing AlgoStrategy

The `MultiStrategy` chains outputs from both sub-strategies:

In [ ]:
type MultiState = EngineState<DefaultGlobalData, MultiStrategyInstrumentData>;

// Each sub-strategy implements AlgoStrategy independently
impl AlgoStrategy for StrategyA {
    type State = MultiState;
    fn generate_algo_orders(
        &self, _: &Self::State,
    ) -> (
        impl IntoIterator<Item = OrderRequestCancel<ExchangeIndex, InstrumentIndex>>,
        impl IntoIterator<Item = OrderRequestOpen<ExchangeIndex, InstrumentIndex>>,
    ) {
        // Your strategy logic here — access state.instruments, state.assets, etc.
        (std::iter::empty(), std::iter::empty())
    }
}

impl AlgoStrategy for StrategyB {
    type State = MultiState;
    fn generate_algo_orders(
        &self, _: &Self::State,
    ) -> (
        impl IntoIterator<Item = OrderRequestCancel<ExchangeIndex, InstrumentIndex>>,
        impl IntoIterator<Item = OrderRequestOpen<ExchangeIndex, InstrumentIndex>>,
    ) {
        (std::iter::empty(), std::iter::empty())
    }
}

// Composite: chain both sub-strategies' outputs
impl AlgoStrategy for MultiStrategy {
    type State = MultiState;

    fn generate_algo_orders(
        &self, state: &Self::State,
    ) -> (
        impl IntoIterator<Item = OrderRequestCancel<ExchangeIndex, InstrumentIndex>>,
        impl IntoIterator<Item = OrderRequestOpen<ExchangeIndex, InstrumentIndex>>,
    ) {
        let (cancels_a, opens_a) = self.strategy_a.generate_algo_orders(state);
        let (cancels_b, opens_b) = self.strategy_b.generate_algo_orders(state);

        // Chain all cancel and open requests from both strategies
        (cancels_a.into_iter().chain(cancels_b), opens_a.into_iter().chain(opens_b))
    }
}

println!("MultiStrategy composes StrategyA + StrategyB via iterator chaining.");

## Strategy Architecture Diagram

```
┌─────────────────────────────────────────────────────┐
│                    MultiStrategy                     │
│                                                     │
│  ┌─────────────┐         ┌─────────────┐           │
│  │ StrategyA    │         │ StrategyB    │           │
│  │ ID: "strat_a"│         │ ID: "strat_b"│           │
│  └──────┬───────┘         └──────┬───────┘           │
│         │                        │                   │
│         ▼                        ▼                   │
│   (cancels_a,              (cancels_b,               │
│    opens_a)                 opens_b)                 │
│         │                        │                   │
│         └────────┬───────────────┘                   │
│                  ▼                                   │
│         .chain() both iterators                      │
└─────────────────────────────────────────────────────┘
                   │
                   ▼
           ┌───────────────┐
           │  RiskManager   │  ← approve/reject
           └───────┬───────┘
                   ▼
           ┌───────────────┐
           │  Execution     │  ← submit to exchange
           └───────────────┘
```

### Key Design Points

- **`StrategyId`**: Each sub-strategy tags its orders so trade fills route back correctly
- **`PositionManager`**: Per-strategy position tracking (not shared across strategies)
- **`TearSheetGenerator`**: Per-strategy performance statistics
- **`InstrumentDataState`**: Custom data must expose `price()` for the engine to use
- **Composition over inheritance**: Strategies are composed via iterator chaining, not trait hierarchy

---
## Part 3: Risk Management


## 1. How Risk Management Fits in the Pipeline

```
Strategy.generate_algo_orders(state)
    │
    ▼
┌──────────────────────────────┐
│       RiskManager.check()    │
│                              │
│  For each order:             │
│    ✓ RiskApproved → execute  │
│    ✗ RiskRefused  → logged   │
└──────────────────────────────┘
    │
    ▼
ExecutionClient (only approved orders)
```

The `RiskManager` receives the full `EngineState` plus all proposed orders,
and returns four iterators: approved cancels, approved opens, refused cancels, refused opens.

In [ ]:
use barter::{
    engine::state::{
        EngineState,
        global::DefaultGlobalData,
        instrument::data::DefaultInstrumentMarketData,
    },
    risk::{
        RiskApproved, RiskManager, RiskRefused,
        check::{
            CheckHigherThan, RiskCheck,
            util::{calculate_abs_percent_difference, calculate_quote_notional},
        },
    },
};
use barter_execution::order::{
    OrderKind,
    request::{OrderRequestCancel, OrderRequestOpen},
};
use barter_instrument::instrument::kind::InstrumentKind;
use rust_decimal::Decimal;
use rust_decimal_macros::dec;

println!("Risk management types imported.");

## 2. Built-in Risk Check Utilities

Barter provides composable check primitives:

In [ ]:
// CheckHigherThan: fails if value exceeds the limit
let max_notional = CheckHigherThan { limit: dec!(50.0) };

// Check passes: 30 < 50
assert!(max_notional.check(&dec!(30.0)).is_ok());
println!("✓ Notional 30 USDT: within limit of 50");

// Check fails: 100 > 50
assert!(max_notional.check(&dec!(100.0)).is_err());
println!("✗ Notional 100 USDT: exceeds limit of 50");

// Notional calculation: quantity * price * contract_size
let notional = calculate_quote_notional(
    dec!(0.5),   // quantity: 0.5 BTC
    dec!(60000),  // price: $60,000
    None,         // contract_size: None for spot (defaults to 1)
).unwrap();
println!("\nNotional for 0.5 BTC @ $60,000 = ${notional}");

// Price deviation check
let deviation = calculate_abs_percent_difference(
    dec!(61000),  // order price
    dec!(60000),  // market price
).unwrap();
println!("Price deviation: {:.4}% from market", deviation * dec!(100));

## 3. Custom RiskManager Implementation

This example implements a `RiskManager` with three checks:
1. **Max notional per order** — reject orders exceeding a USDT threshold
2. **Market order price deviation** — reject market orders too far from current price
3. **Instrument kind filter** — reject orders for unsupported instrument types (e.g., Options)

In [ ]:
use std::marker::PhantomData;

type DefaultState = EngineState<DefaultGlobalData, DefaultInstrumentMarketData>;

/// Custom risk manager with configurable limits
#[derive(Debug, Clone)]
pub struct CustomRiskManager {
    /// Maximum quote-currency notional per order
    pub max_notional_per_order: CheckHigherThan<Decimal>,
    /// Maximum % deviation from market price for MARKET orders
    pub max_market_price_deviation: CheckHigherThan<Decimal>,
}

impl Default for CustomRiskManager {
    fn default() -> Self {
        Self {
            max_notional_per_order: CheckHigherThan { limit: dec!(50.0) },       // 50 USDT
            max_market_price_deviation: CheckHigherThan { limit: dec!(0.1) },     // 10%
        }
    }
}

println!("CustomRiskManager defined with:");
let rm = CustomRiskManager::default();
println!("  Max notional per order: {} USDT", rm.max_notional_per_order.limit);
println!("  Max market price deviation: {}%", rm.max_market_price_deviation.limit * dec!(100));

## 4. Implementing the RiskManager Trait

The `check()` method receives all proposed orders and must categorise each as
approved or refused. The engine only executes approved orders.

In [ ]:
impl RiskManager for CustomRiskManager {
    type State = DefaultState;

    fn check(
        &self,
        state: &Self::State,
        cancels: impl IntoIterator<Item = OrderRequestCancel>,
        opens: impl IntoIterator<Item = OrderRequestOpen>,
    ) -> (
        impl IntoIterator<Item = RiskApproved<OrderRequestCancel>>,
        impl IntoIterator<Item = RiskApproved<OrderRequestOpen>>,
        impl IntoIterator<Item = RiskRefused<OrderRequestCancel>>,
        impl IntoIterator<Item = RiskRefused<OrderRequestOpen>>,
    ) {
        // Always approve cancels (no risk in cancelling)
        let approved_cancels: Vec<_> = cancels
            .into_iter()
            .map(RiskApproved::new)
            .collect();

        let (mut approved_opens, mut refused_opens) = (Vec::new(), Vec::new());

        for request in opens {
            let instrument_state = state
                .instruments
                .instrument_index(&request.key.instrument);

            // Check 1: Reject unsupported instrument kinds
            if let InstrumentKind::Option(_) = instrument_state.instrument.kind {
                refused_opens.push(RiskRefused::new(
                    request,
                    "Options orders not supported by this risk manager",
                ));
                continue;
            }

            // Check 2: Max notional
            let notional = calculate_quote_notional(
                request.state.quantity,
                request.state.price,
                instrument_state.instrument.kind.contract_size(),
            ).expect("notional overflow");

            if self.max_notional_per_order.check(&notional).is_err() {
                refused_opens.push(RiskRefused::new(
                    request,
                    "Notional exceeds max_notional_per_order limit",
                ));
                continue;
            }

            // Check 3: Market order price deviation (only for MARKET orders)
            if OrderKind::Market == request.state.kind {
                if let Some(market_price) = instrument_state.data.price() {
                    let deviation = calculate_abs_percent_difference(
                        request.state.price,
                        market_price,
                    ).expect("price deviation overflow");

                    if self.max_market_price_deviation.check(&deviation).is_err() {
                        refused_opens.push(RiskRefused::new(
                            request,
                            "Market order price too far from current market",
                        ));
                        continue;
                    }
                }
            }

            // All checks passed
            approved_opens.push(RiskApproved::new(request));
        }

        (approved_cancels, approved_opens, std::iter::empty(), refused_opens)
    }
}

println!("CustomRiskManager trait implementation complete.");
println!("\nOrder flow: Strategy → RiskManager.check() → approved orders → Execution");

## 5. Risk Check Composition Patterns

### Available Check Types

| Check | Purpose | Example |
|-------|---------|--------|
| `CheckHigherThan<T>` | Reject if value > limit | Max notional, max deviation |

### Utility Functions

| Function | Purpose |
|----------|--------|
| `calculate_quote_notional(qty, price, contract_size)` | Compute order value in quote currency |
| `calculate_abs_percent_difference(a, b)` | Compute \|a-b\|/b as a decimal fraction |

### Recommended Risk Checks for Production

1. **Max notional per order** — prevents fat-finger errors
2. **Max notional per instrument** — limits exposure per market
3. **Max total portfolio exposure** — global position limit
4. **Market order price deviation** — prevents slippage on stale prices
5. **Order rate limiting** — prevents runaway strategy loops
6. **Instrument kind whitelist** — only trade supported derivatives
7. **Balance sufficiency** — check free balance before opening

### Recoverability

Risk refusals implement the `Unrecoverable` trait:
- **Recoverable**: Strategy can try again next tick (e.g., price deviation)
- **Unrecoverable**: Fatal condition that should shut down the engine

---
## Part 4: Trading Statistics & Performance Analysis


## 1. Define Instruments and Initial State

`IndexedInstruments` is the central registry mapping exchanges, assets, and
instruments to compact integer indices for O(1) lookups.

In [ ]:
use barter::{
    engine::state::{
        EngineState, global::DefaultGlobalData,
        instrument::data::DefaultInstrumentMarketData,
        position::PositionExited, trading::TradingState,
    },
    statistic::{summary::TradingSummaryGenerator, time::Annual365},
};
use barter_execution::{
    balance::{AssetBalance, Balance},
    trade::{AssetFees, TradeId},
};
use barter_instrument::{
    Side, Underlying,
    asset::{AssetIndex, QuoteAsset},
    exchange::ExchangeId,
    index::IndexedInstruments,
    instrument::{Instrument, InstrumentIndex},
};
use barter_integration::collection::snapshot::Snapshot;
use chrono::{DateTime, Days, Utc};
use rust_decimal::Decimal;
use rust_decimal_macros::dec;
use smol_str::SmolStr;

// Risk-free rate for Sharpe/Sortino calculations
const RISK_FREE_RETURN: Decimal = dec!(0.05);

// Build instrument index: BTC/USDT and ETH/USDT on Binance Spot
let instruments = IndexedInstruments::builder()
    .add_instrument(Instrument::spot(
        ExchangeId::BinanceSpot,
        "binance_spot_btc_usdt",
        "BTCUSDT",
        Underlying::new("btc", "usdt"),
        None,
    ))
    .add_instrument(Instrument::spot(
        ExchangeId::BinanceSpot,
        "binance_spot_eth_usdt",
        "ETHUSDT",
        Underlying::new("eth", "usdt"),
        None,
    ))
    .build();

println!("Indexed {} instruments", instruments.instruments().len());
println!("Indexed {} assets", instruments.assets().len());

## 2. Build Engine State with Balances

The `EngineState` builder initialises the state with starting balances per
exchange and asset. This is the same state structure used by the live Engine.

In [ ]:
let time_start = Utc::now();

let state = EngineState::builder(&instruments, DefaultGlobalData::default(), |_| {
    DefaultInstrumentMarketData::default()
})
.time_engine_start(time_start)
.trading_state(TradingState::Enabled)
.balances([
    (ExchangeId::BinanceSpot, "usdt", Balance::new(dec!(10_000.0), dec!(10_000.0))),
    (ExchangeId::BinanceSpot, "btc",  Balance::new(dec!(0.1), dec!(0.1))),
    (ExchangeId::BinanceSpot, "eth",  Balance::new(dec!(1.0), dec!(1.0))),
])
.build();

println!("EngineState built with starting balances.");

## 3. Initialise TradingSummaryGenerator

The generator tracks balance changes and closed positions over time, then
computes portfolio-level and per-instrument statistics.

In [ ]:
let mut summary_gen = TradingSummaryGenerator::init(
    RISK_FREE_RETURN,
    time_start,
    time_start,
    &state.instruments,
    &state.assets,
);

println!("TradingSummaryGenerator initialised.");
println!("Risk-free rate: {RISK_FREE_RETURN}");

## 4. Feed Synthetic Trading Events

We simulate a series of trades:
1. Buy BTC (spend 1000 USDT) → sell for profit (+2000)
2. Buy BTC again → sell for profit (+1000)
3. Buy BTC → sell at a loss (-2000)
4. Buy ETH → sell at a loss (-1000)
5. Buy ETH → sell for small profit (+500)

In [ ]:
// Helper to create a balance update
fn balance_update(usdt_total: Decimal, day_offset: u64, base_time: DateTime<Utc>) -> Snapshot<AssetBalance<AssetIndex>> {
    Snapshot::new(AssetBalance {
        asset: AssetIndex(2), // usdt
        balance: Balance::new(usdt_total, usdt_total),
        time_exchange: base_time.checked_add_days(Days::new(day_offset)).unwrap(),
    })
}

// Helper to create a closed position
fn closed_position(
    instrument: InstrumentIndex, pnl: Decimal,
    qty: Decimal, day_enter: u64, day_exit: u64,
    trade_ids: Vec<&str>, base_time: DateTime<Utc>,
) -> PositionExited<QuoteAsset, InstrumentIndex> {
    PositionExited {
        instrument,
        side: Side::Buy,
        price_entry_average: dec!(1.0),
        quantity_abs_max: qty,
        pnl_realised: pnl,
        fees_enter: AssetFees::default(),
        fees_exit: AssetFees::default(),
        time_enter: base_time.checked_add_days(Days::new(day_enter)).unwrap(),
        time_exit: base_time.checked_add_days(Days::new(day_exit)).unwrap(),
        trades: trade_ids.into_iter().map(|id| TradeId(SmolStr::new(id))).collect(),
    }
}

let btc = InstrumentIndex(0);
let eth = InstrumentIndex(1);

// Trade 1: BTC buy (10000 -> 9000 USDT), then sell for profit (9000 -> 12000)
summary_gen.update_from_balance(balance_update(dec!(9000), 1, time_start).as_ref());
summary_gen.update_from_balance(balance_update(dec!(12000), 2, time_start).as_ref());
summary_gen.update_from_position(&closed_position(btc, dec!(2000), dec!(1000), 1, 2, vec!["1","2"], time_start));
println!("Trade 1 (BTC): +2000 USDT profit");

// Trade 2: BTC buy (12000 -> 10000), sell (10000 -> 13000)
summary_gen.update_from_balance(balance_update(dec!(10000), 2, time_start).as_ref());
summary_gen.update_from_balance(balance_update(dec!(13000), 3, time_start).as_ref());
summary_gen.update_from_position(&closed_position(btc, dec!(1000), dec!(2000), 2, 3, vec!["3","4"], time_start));
println!("Trade 2 (BTC): +1000 USDT profit");

// Trade 3: BTC loss (13000 -> 8000 -> 11000)
summary_gen.update_from_balance(balance_update(dec!(8000), 4, time_start).as_ref());
summary_gen.update_from_balance(balance_update(dec!(11000), 5, time_start).as_ref());
summary_gen.update_from_position(&closed_position(btc, dec!(-2000), dec!(2000), 4, 5, vec!["5","6"], time_start));
println!("Trade 3 (BTC): -2000 USDT loss");

// Trade 4: ETH loss (11000 -> 6000 -> 5000 -> 10000)
summary_gen.update_from_balance(balance_update(dec!(6000), 6, time_start).as_ref());
summary_gen.update_from_balance(balance_update(dec!(5000), 7, time_start).as_ref());
summary_gen.update_from_balance(balance_update(dec!(10000), 8, time_start).as_ref());
summary_gen.update_from_position(&closed_position(eth, dec!(-1000), dec!(6000), 6, 8, vec!["7","8","9"], time_start));
println!("Trade 4 (ETH): -1000 USDT loss");

// Trade 5: ETH small profit (10000 -> 7000 -> 10500)
summary_gen.update_from_balance(balance_update(dec!(7000), 10, time_start).as_ref());
summary_gen.update_from_balance(balance_update(dec!(10500), 11, time_start).as_ref());
summary_gen.update_from_position(&closed_position(eth, dec!(500), dec!(6000), 10, 11, vec!["10","11"], time_start));
println!("Trade 5 (ETH): +500 USDT profit");

println!("\nAll synthetic events fed.");

## 5. Generate & Display Trading Summary

Call `.generate(Annual365)` for crypto-style 24/7 annualisation, or
`.generate(Annual252)` for traditional equity-style (252 trading days).

In [ ]:
let summary = summary_gen.generate(Annual365);

// Print the full summary table to the terminal
summary.print_summary();

## Available Metrics

The `TradingSummary` includes per-instrument and portfolio-wide statistics:

| Metric | Description |
|--------|-------------|
| **Total PnL** | Net realised profit/loss |
| **Win Rate** | Percentage of profitable trades |
| **Sharpe Ratio** | Risk-adjusted return (vs risk-free rate) |
| **Sortino Ratio** | Downside-risk-adjusted return |
| **Calmar Ratio** | Return / max drawdown |
| **Max Drawdown** | Largest peak-to-trough decline |
| **Rate of Return** | Annualised return percentage |
| **Profit Factor** | Gross profit / gross loss |

### Time Intervals

- `Annual365` — 24/7 crypto markets (365 days/year)
- `Annual252` — traditional equity markets (252 trading days/year)
- `Daily` — daily statistics without annualisation